<a href="https://colab.research.google.com/github/akilaam/Article-recommendation/blob/main/Research_LDA_with_trained_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyPDF2

In [ ]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 7.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [ ]:
import os
import pandas as pd
import PyPDF2
import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive
from gensim.models.coherencemodel import CoherenceModel
from gensim import corpora
from gensim.utils import simple_preprocess

# Mount Google Drive
drive.mount('/content/drive')

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# 🔹 Function to extract text from PDFs
def extract_paragraphs_from_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        paragraphs = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                paragraphs.extend(text.split(r'\n|\r\n'))
        return paragraphs

# 🔹 Function to preprocess text (Tokenization, Stopword removal)
def preprocess_text(text):
    tokens = word_tokenize(text.lower())  # Convert to lowercase and tokenize
    tokens = [word for word in tokens if word.isalpha()]  # Remove punctuation
    stop_words = set(stopwords.words('english'))
    return ' '.join([word for word in tokens if word not in stop_words])

# 🔹 Load Training Data (CSV File)
csv_path = '/content/drive/My Drive/Recommendation system/magazines/training_data.csv'
df = pd.read_csv(csv_path)

# 🔹 Ensure the CSV has 'Article' and 'Topic' columns
if 'Article' not in df.columns or 'Topic ' not in df.columns:
    raise ValueError("CSV file must contain 'Article' and 'Topic' columns.")

# 🔹 Preprocess CSV Text Data
df['processed_text'] = df['Article'].astype(str).apply(preprocess_text)

# 🔹 Convert CSV text into TF-IDF features for topic modeling
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(df['processed_text'])

# 🔹 Train LSA Model using the CSV data
num_topics = len(df['Topic '].unique())  # Number of topics = unique topics in CSV
lsa_model = TruncatedSVD(n_components=num_topics)
lsa_topic_matrix = lsa_model.fit_transform(X_train)

# 🔹 Map Topics from CSV to LSA Model
topic_mapping = {i: topic for i, topic in enumerate(df['Topic '].unique())}

print("\n🔍 **Topics in Training Data:**")
for i, topic in topic_mapping.items():
    print(f"**LSA Topic {i+1}:** {topic}")

# 🔹 Path to PDF Folder
pdf_folder_path = '/content/drive/My Drive/Recommendation system/magazines/Data Science/Data Science in Science'
os.chdir(pdf_folder_path)

# 🔹 Process PDFs and Assign Topics
dict_paragraphs = []
all_paragraphs = []

for filename in os.listdir(pdf_folder_path):
    if filename.endswith('.pdf'):
        pdf_path = os.path.join(pdf_folder_path, filename)
        journal_name = filename[:-4]

        paragraphs = extract_paragraphs_from_pdf(pdf_path)
        for paragraph in paragraphs:
            if len(paragraph.split()) > 50:  # Filter out short paragraphs
                preprocessed_text = preprocess_text(paragraph)
                dict_paragraphs.append({
                    "journal_name": journal_name,
                    "preprocessed_paragraph": preprocessed_text
                })
                all_paragraphs.append(preprocessed_text)

# 🔹 Convert PDF text into TF-IDF features
X_pdf = vectorizer.transform(all_paragraphs)  # Use the same TF-IDF model
pdf_topic_matrix = lsa_model.transform(X_pdf)  # Apply trained LSA model

# 🔹 Assign Real Topics from Training Data to PDF Paragraphs
print("\n **PDF Paragraphs & Assigned Topics:**\n")

for i, entry in enumerate(dict_paragraphs):
    similarities = cosine_similarity([pdf_topic_matrix[i]], lsa_topic_matrix).flatten()
    topic_index = np.argmax(similarities)  # Get the closest topic from training data
    assigned_topic = topic_mapping.get(topic_index, topic)

    print(f"File:{entry['journal_name']}")
    print(f"Paragraph:\n{entry['preprocessed_paragraph']}")
    print(f"Assigned Topic:{topic_index}")
    print("-" * 80)


Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



🔍 **Topics in Training Data:**
**LSA Topic 1:** Statistics
**LSA Topic 2:** Machine Learning
**LSA Topic 3:** Data Analysis
**LSA Topic 4:** Data Engineering
**LSA Topic 5:** Data Visualization
**LSA Topic 6:** Business Intelligence (BI)

 **PDF Paragraphs & Assigned Topics:**

File:00-Rewiring Dynamics of Functional Connectomes during Motor-Skill Learning
Paragraph:
full terms conditions access use found https data science science issn print online journal homepage https rewiring dynamics functional connectomes learning saber meamardoost mahasweta bhattacharya eun jung hwang chi ren linbing wang claudia mewes ying zhang takaki komiyama rudiyanto gunawan cite article saber meamardoost mahasweta bhattacharya eun jung hwang chi ren linbing wang claudia mewes ying zhang takaki komiyama rudiyanto gunawan rewiring dynamics functional connectomes learning data science science doi link article https author published license taylor francis group llc view supplementary material published onlin

In [ ]:
topic_mapping


{0: 'Statistics',
 1: 'Machine Learning',
 2: 'Data Analysis',
 3: 'Data Engineering',
 4: 'Data Visualization',
 5: 'Business Intelligence (BI)'}

In [ ]:
for i, entry in enumerate(dict_paragraphs):
    similarities = cosine_similarity([pdf_topic_matrix[i]], lsa_topic_matrix).flatten()
    topic_index = np.argmax(similarities)  # Get the closest topic from training data
    assigned_topic = topic_mapping.get(topic_index,topic)

    print(f"File:{entry['journal_name']}")
    print(f"Paragraph:\n{entry['preprocessed_paragraph']}")
    print(f"Assigned Topic:{topic_index}--{assigned_topic}")
    print("-" * 80)

File:00-Rewiring Dynamics of Functional Connectomes during Motor-Skill Learning
Paragraph:
full terms conditions access use found https data science science issn print online journal homepage https rewiring dynamics functional connectomes learning saber meamardoost mahasweta bhattacharya eun jung hwang chi ren linbing wang claudia mewes ying zhang takaki komiyama rudiyanto gunawan cite article saber meamardoost mahasweta bhattacharya eun jung hwang chi ren linbing wang claudia mewes ying zhang takaki komiyama rudiyanto gunawan rewiring dynamics functional connectomes learning data science science doi link article https author published license taylor francis group llc view supplementary material published online oct submit article journal article views view related articles view crossmark data
Assigned Topic:39--Business Intelligence (BI)
--------------------------------------------------------------------------------
File:00-Rewiring Dynamics of Functional Connectomes during Motor-Ski

In [ ]:
 import textwrap

# Associate each paragraph with its most relevant topic
for i, entry in enumerate(dict_paragraphs):
    topic_most_pr = lsa_topic_matrix[i].argmax()

    print(f"\nMagazine: {entry['journal_name']}")
    print(f"Assigned Topic: {topic_most_pr}\n")

    # Format paragraph with proper wrapping
    print("Paragraph:")
    print(textwrap.fill(entry['preprocessed_paragraph'], width=80))  # Wrap text at 80 characters

    print("\n" + "-" * 100)  # Separator for readability




# 🔹 Assign Real Topics from Training Data to PDF Paragraphs
print("\n📚 **PDF Paragraphs & Assigned Topics:**\n")

for i, entry in enumerate(dict_paragraphs):
    similarities = cosine_similarity([pdf_topic_matrix[i]], lsa_topic_matrix).flatten()
    topic_index = np.argmax(similarities)  # Get the closest topic from training data
    assigned_topic = topic_mapping.get(topic_index, "Unknown Topic")

    print(f"File:{entry['journal_name']}")
    print(f"Paragraph:\n{entry['preprocessed_paragraph']}")
    print(f"Assigned Topic:{assigned_topic}")
    print("-" * 80)


Magazine: 00-Rewiring Dynamics of Functional Connectomes during Motor-Skill Learning
Assigned Topic: 2

Paragraph:
full terms conditions access use found https data science science issn print
online journal homepage https rewiring dynamics functional connectomes learning
saber meamardoost mahasweta bhattacharya eun jung hwang chi ren linbing wang
claudia mewes ying zhang takaki komiyama rudiyanto gunawan cite article saber
meamardoost mahasweta bhattacharya eun jung hwang chi ren linbing wang claudia
mewes ying zhang takaki komiyama rudiyanto gunawan rewiring dynamics functional
connectomes learning data science science doi link article https author
published license taylor francis group llc view supplementary material published
online oct submit article journal article views view related articles view
crossmark data

----------------------------------------------------------------------------------------------------

Magazine: 00-Rewiring Dynamics of Functional Connectomes during Mot

IndexError: index 58 is out of bounds for axis 0 with size 58

In [ ]:
print(f"pdf_topic_matrix shape: {pdf_topic_matrix.shape}")
print(f"lsa_topic_matrix shape: {lsa_topic_matrix.shape}")
print(f"Current index: {i}")


pdf_topic_matrix shape: (211, 6)
lsa_topic_matrix shape: (58, 6)
Current index: 58
